# Exercise 4b — LoRA Adaptation of DINOv3 on Microscopy Images


**CAS Deep Learning — Computer Vision**

## Learning objectives
After this exercise you should be able to:
- Explain why full fine-tuning of a large model is impractical with limited data
- Apply LoRA (Low-Rank Adaptation) to a large Vision Transformer using the `peft` library
- Compare a linear probe against LoRA adaptation in a domain-shifted setting
- Reason about the accuracy–parameter-count trade-off

## Context
Exercise 4 ended with partial fine-tuning of a 28M-parameter ConvNeXt on camera trap images.  
Here we push further: the model has **~600M parameters**, the dataset has **only ~400 training images**,  
and the domain shifts drastically — from natural ImageNet photos to **histology microscopy patches**  
of breast cancer tissue. Full fine-tuning is out of the question.  
LoRA lets us adapt the attention layers with a tiny fraction of trainable parameters.

## Dataset — AMi-Br
The [AMi-Br dataset](https://github.com/DeepMicroscopy/AMi-Br) contains 128×128 px crops of  
mitotic figures (cell divisions) in breast cancer histology slides.  
**Task:** binary classification — *normal* vs *atypical* mitotic figures.  
Atypical figures are a key diagnostic marker for aggressive cancer.


## Note
- **Good GPU strongly recommended** (Colab: Runtime → Change runtime type → T4 GPU).  
  Feature extraction (Part 2) works on CPU; LoRA training (Part 3) is very slow without GPU.

**Fill in code where you see `# YOUR CODE HERE`.**

## Setup Colab / Local Paths

The following setup sets the default data path: Make sure to check and adapt `DATA_PATH`.

If working in Colab: **Save the notebook to your personal Google Drive to persist changes**.

Packages not in the Colab environment will be installed as well.

In [ ]:
import sys
from pathlib import Path

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

IN_COLAB = "google.colab" in sys.modules
print(f"In Colab: {IN_COLAB}")


if IN_COLAB:
    # Mirrors requirements.txt — keep them in sync.
    print("Installing additional packages to Colab Environment")
    !pip install -q timm peft

    print("Mounting Drive")
    from google.colab import drive

    ROOT_PATH = Path("/content/drive")
    drive.mount(str(ROOT_PATH))
    DATA_PATH = ROOT_PATH.joinpath("MyDrive/cas-dl-compvis")
else:
    import pyrootutils

    ROOT_PATH = pyrootutils.setup_root(
        search_from=".",
        indicator="pyproject.toml",
        project_root_env_var=True,
        dotenv=True,
        pythonpath=True,
        cwd=True,
    )

    DATA_PATH = ROOT_PATH.joinpath("data")

print(f"Creating {DATA_PATH} if it does not exist")
DATA_PATH.mkdir(parents=True, exist_ok=True)

## Imports

In [ ]:
import random
import time
import zipfile
from collections import defaultdict

import matplotlib.pyplot as plt
import requests
import timm
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm.notebook import tqdm

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device} | Colab: {IN_COLAB}")
if device == "cpu":
    print("⚠  GPU not detected. LoRA training (Part 3) will be very slow on CPU.")
    print("  On Google Colab: Runtime → Change runtime type → T4 GPU")

## Parameters

In [ ]:
random.seed(123)
torch.manual_seed(123)

## Part 1 — AMi-Br: Mitotic Figure Dataset

Histology slides from breast cancer biopsies are scanned at high magnification.  
Pathologists look for *mitotic figures* (cells actively dividing) and mark atypical ones —  
shapes that indicate uncontrolled, aggressive cell division.  
The AMi-Br dataset provides 128×128 px patches labelled as **normal** or **atypical**.

The patches are organised in two subdirectories — perfectly suited for `torchvision.datasets.ImageFolder`:
```
patches/
├── atypical/   ← label 0
└── normal/     ← label 1
```

In [ ]:
def download_ami_br(data_path: Path) -> Path:
    """Download and extract the AMi-Br patch dataset from GitHub."""
    out_dir = data_path / "ami_br"
    if (out_dir / "patches").exists():
        print(f"Dataset already present: {out_dir}")
        return out_dir
    url = "https://github.com/DeepMicroscopy/AMi-Br/archive/refs/heads/main.zip"
    print("Downloading AMi-Br from GitHub (~50 MB)...")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    zip_path = data_path / "ami_br.zip"
    zip_path.write_bytes(r.content)
    print("Extracting...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(data_path)
    (data_path / "AMi-Br-main").rename(out_dir)
    zip_path.unlink()
    print(f"Done. Extracted to: {out_dir}")
    return out_dir


AMI_BR_PATH = download_ami_br(DATA_PATH)
print(f"Patches: {AMI_BR_PATH / 'patches'}")

In [ ]:
# ImageNet normalisation — DINOv3 was pretrained on ImageNet
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Evaluation transforms: resize 128→224 (DINOv3 ViT expects 224x224), normalise
eval_transforms = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ]
)

# Training transforms: add light augmentation
train_transforms = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ]
)

# Load full dataset — ImageFolder uses subdirectory names as class labels
full_dataset = ImageFolder(root=str(AMI_BR_PATH / "patches"), transform=eval_transforms)
CLASS_NAMES = full_dataset.classes  # ['atypical', 'normal']
NUM_CLASSES = len(CLASS_NAMES)

# Count images per class
class_counts = defaultdict(int)
for _, label in full_dataset.samples:
    class_counts[CLASS_NAMES[label]] += 1

print(f"Classes: {CLASS_NAMES}")
print(f"Total images: {len(full_dataset):,}")
for cls, count in class_counts.items():
    print(f"  {cls}: {count:,} images")
print(
    "\nNote the class imbalance — normal mitoses are far more common than atypical ones."
)

In [ ]:
# Visualise sample patches from each class
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
_ = fig.suptitle(
    "Sample AMi-Br Patches — 128x128 px breast cancer histology crops", fontsize=12
)

for row, cls in enumerate(CLASS_NAMES):
    indices = [
        i for i, (_, lbl) in enumerate(full_dataset.samples) if CLASS_NAMES[lbl] == cls
    ][:6]
    for col, idx in enumerate(indices):
        img_path = full_dataset.samples[idx][0]
        img = plt.imread(img_path)
        _ = axes[row, col].imshow(img)
        _ = axes[row, col].set_title(cls, fontsize=9)
        _ = axes[row, col].axis("off")

_ = plt.tight_layout()
plt.show()

### Task 1 — Balanced Subset & Stratified Train/Val Split

The dataset is heavily imbalanced — there are far more *normal* patches than *atypical* ones.  
For this exercise we create a **balanced** subset capped at `MAX_PER_CLASS` images per class,  
then split it into stratified train and validation sets.

**Your task:**
1. Build `balanced_indices` — for each class, randomly sample up to `MAX_PER_CLASS` image indices.
2. Build `balanced_labels` — the corresponding label for each index in `balanced_indices`.
3. Split into `train_indices` and `val_indices` using `train_test_split` with `test_size=0.3`  
   and `stratify=balanced_labels`.

<details>
<summary>Hint</summary>

```python
# Group indices by class
class_indices = defaultdict(list)
for idx, (_, label) in enumerate(full_dataset.samples):
    class_indices[CLASS_NAMES[label]].append(idx)

# Sample up to MAX_PER_CLASS from each class
balanced_indices = []
for cls in CLASS_NAMES:
    idxs = class_indices[cls]
    sampled = random.sample(idxs, min(len(idxs), MAX_PER_CLASS))
    balanced_indices.extend(sampled)
```

</details>

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
# Verify split
train_labels = [full_dataset.targets[i] for i in train_indices]
val_labels = [full_dataset.targets[i] for i in val_indices]

assert len(train_indices) > 0 and len(val_indices) > 0
assert set(train_labels) == {0, 1} and set(val_labels) == {0, 1}
print("OK")

print(f"Train: {len(train_indices)} images")
for cls_idx, cls in enumerate(CLASS_NAMES):
    count = sum(1 for label in train_labels if label == cls_idx)
    print(f"  {cls}: {count}")
print(f"Val:   {len(val_indices)} images")
for cls_idx, cls in enumerate(CLASS_NAMES):
    count = sum(1 for label in val_labels if label == cls_idx)
    print(f"  {cls}: {count}")

In [ ]:
BATCH_SIZE = 16

# For feature extraction (linear probe): eval transforms on both splits
train_subset_eval = Subset(full_dataset, train_indices)
val_subset = Subset(full_dataset, val_indices)

# For LoRA training: augmented transforms on train split
full_dataset_aug = ImageFolder(
    root=str(AMI_BR_PATH / "patches"), transform=train_transforms
)
train_subset_aug = Subset(full_dataset_aug, train_indices)

train_loader_eval = DataLoader(
    train_subset_eval, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
train_loader_aug = DataLoader(
    train_subset_aug, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)

print(f"Train batches: {len(train_loader_eval)} | Val batches: {len(val_loader)}")

## Part 2 — DINOv3 as a Fixed Feature Extractor (Linear Probe)

We load `vit_huge_plus_patch16_dinov3.lvd1689m` from timm — a ~600M parameter Vision Transformer  
pretrained with the DINOv3 self-supervised recipe on the LVD-1689M dataset.

**The domain gap is large:** DINOv3 saw natural photographs; we're classifying histology patches.  
A *linear probe* answers the question: **how much does the frozen backbone already know?**  
It trains only a single linear layer on top of the frozen features — no fine-tuning at all.

This is also fast: we extract features once and cache them, then train on the cached tensors.

In [ ]:
MODEL_NAME = "vit_huge_plus_patch16_dinov3.lvd1689m"

print(f"Loading {MODEL_NAME} ...")
print("(First run: downloads 3.36 GB of weights — takes a minute on Colab)")

# num_classes=0 → model returns the pooled CLS-token embedding instead of logits
encoder = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0)
encoder = encoder.to(device)
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False

# Probe output shape dynamically (handles any ViT-Huge+ variant)
with torch.no_grad():
    _dummy = torch.randn(1, 3, 224, 224, device=device)
    EMBED_DIM = encoder(_dummy).shape[1]

total_params = sum(p.numel() for p in encoder.parameters())
print("\nLoaded.")
print(f"Total parameters:  {total_params:,}  (~{total_params / 1e6:.0f}M)")
print(f"Embedding dim:     {EMBED_DIM}")
print(
    f"\nFull fine-tuning of {total_params / 1e6:.0f}M params with only ~{len(train_indices)} images"
)
print("might overfit — this motivates smarter adaptation strategies.")

In [ ]:
# Extract CLS-token embeddings for all images in a DataLoader
# timm with num_classes=0 returns the global pooled CLS token as (B, EMBED_DIM)
@torch.no_grad()
def extract_features(model, data_loader, device):
    model.eval()
    all_feats, all_labels = [], []
    for images, labels in tqdm(data_loader, desc="Extracting features"):
        feats = model(images.to(device))  # (B, EMBED_DIM)
        all_feats.append(feats.cpu())
        all_labels.append(labels)
    return torch.cat(all_feats).numpy(), torch.cat(all_labels).numpy()


print("Extracting features (train)...")
t0 = time.time()
train_feats, train_feat_labels = extract_features(encoder, train_loader_eval, device)
print("Extracting features (val)...")
val_feats, val_feat_labels = extract_features(encoder, val_loader, device)
feat_time = time.time() - t0

print(f"\nTrain features: {train_feats.shape}")
print(f"Val features:   {val_feats.shape}")
print(f"Extraction time: {feat_time:.1f}s")

In [ ]:
assert train_feats.shape == (len(train_indices), EMBED_DIM), (
    f"Expected ({len(train_indices)}, {EMBED_DIM}), got {train_feats.shape}"
)
assert val_feats.shape == (len(val_indices), EMBED_DIM)
print("OK")

### Task 2 — Train the Linear Probe

Train a **single linear layer** (`nn.Linear(EMBED_DIM, NUM_CLASSES)`) on the cached features.  
No images pass through the encoder during training — only the cached embeddings.  
This makes training very fast.

**Your task:** implement the training loop:
- 100 epochs
- `nn.CrossEntropyLoss`, `Adam` with `lr=1e-3`
- Track `train_loss`, `train_acc`, `val_acc` in `lp_history`
- Report accuracy every 5 epochs
- Store the final val accuracy in `lp_acc` and training time in `lp_time`

<details>
<summary>Hint: wrap features in a TensorDataset</summary>

```python
train_feat_ds = TensorDataset(
    torch.from_numpy(train_feats),
    torch.from_numpy(train_feat_labels).long(),
)
train_feat_dl = DataLoader(train_feat_ds, batch_size=64, shuffle=True)
```

</details>

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
assert 0.0 < lp_acc <= 1.0, f"Accuracy should be between 0 and 1, got {lp_acc}"
assert len(lp_history["train_loss"]) == NUM_EPOCHS_LP
print(f"OK — linear probe accuracy: {lp_acc:.1%}")

In [ ]:
# Collect predictions for classification report
preds_lp = []
linear_probe.eval()
with torch.no_grad():
    for feats, _ in val_feat_dl:
        preds_lp.extend(linear_probe(feats.to(device)).argmax(1).cpu().tolist())

print(classification_report(val_feat_labels, preds_lp, target_names=CLASS_NAMES))

epochs = range(1, NUM_EPOCHS_LP + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
_ = axes[0].plot(epochs, lp_history["train_loss"], "o-", markersize=3)
_ = axes[0].set(xlabel="Epoch", ylabel="Loss", title="Linear Probe — Training Loss")
_ = axes[0].grid(alpha=0.3)
_ = axes[1].plot(epochs, lp_history["train_acc"], "o-", markersize=3, label="Train")
_ = axes[1].plot(epochs, lp_history["val_acc"], "o-", markersize=3, label="Val")
_ = axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="Linear Probe — Accuracy")
_ = axes[1].legend()
_ = axes[1].grid(alpha=0.3)
_ = plt.tight_layout()
plt.show()

**Question:** The model has never seen histology images, yet the linear probe likely achieves reasonable accuracy.
What does this tell us about DINOv3's pretrained features?

<details>
<summary>Click to reveal answer</summary>

DINOv3 learns *general visual structure* — textures, shapes, spatial patterns — that transfer  
across domains even without task-specific supervision.  
The self-supervised objective (predicting masked patches or distorted views) forces the model  
to build rich representations that cluster visually similar images together.  
Even for histology, where colours and textures are very different from natural images,  
the frozen features are already partially informative.

A linear probe *cannot change the features* — it can only rotate and scale them.  
If the features don't separate the classes, a linear probe will fail regardless of training time.  
That's why a strong linear probe result is a good early signal of transfer quality.

</details>

## Part 3 — LoRA Adaptation

The linear probe keeps the backbone completely frozen.  
What if the features aren't quite right for histology — can we *adapt* the backbone  
without fine-tuning all 600M parameters?

### How LoRA works

A weight matrix **W** ∈ ℝ^(d×k) is large and expensive to update.  
LoRA freezes **W** and adds a low-rank update:

$$W' = W + \Delta W = W + BA$$

where **B** ∈ ℝ^(d×r) and **A** ∈ ℝ^(r×k), with rank **r ≪ min(d, k)**.  
Only **A** and **B** are trained. For r=8 on a 1280×3840 QKV matrix:  
original: **4,915,200** params → LoRA: **40,960** params → **120× fewer**.

We inject LoRA into the attention **QKV** projection of every transformer block.  

### timm vs HF naming

timm ViT uses a **combined** QKV matrix (`attn.qkv`) — one linear layer that projects to  
Q, K, V simultaneously.  
HuggingFace transformers splits it into three separate layers (`query`, `key`, `value`).  
The LoRA target module name therefore differs between the two loading approaches.

In [ ]:
# Load a fresh backbone for LoRA (separate from the frozen encoder above)
print(f"Loading fresh {MODEL_NAME} for LoRA...")
lora_backbone = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0)
lora_backbone = lora_backbone.to(device)

# Configure LoRA
# target_modules="qkv": inject adapters into the combined QKV projection (timm ViT naming)
# r=8: low-rank dimension — controls adapter size
# lora_alpha=16: scaling factor (effective update scale ∝ alpha/r = 2.0)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["qkv"],  # timm ViT: combined QKV matrix (not HF's "query"/"value")
    lora_dropout=0.05,
    bias="none",
)

# get_peft_model works on ANY nn.Module, not just HF PreTrainedModel
lora_backbone = get_peft_model(lora_backbone, lora_config)
lora_backbone.print_trainable_parameters()
# Expected: ~1.3M trainable out of ~800M total  (~0.15%)

In [ ]:
# Sanity check: LoRA adapters must have been injected
lora_trainable = sum(p.numel() for p in lora_backbone.parameters() if p.requires_grad)
assert lora_trainable > 0, (
    "LoRA injected no trainable params — target_modules didn't match any layer.\n"
    "Inspect available attention layers with:\n"
    "  for n, m in lora_backbone.named_modules():\n"
    "      if isinstance(m, torch.nn.Linear) and 'attn' in n: print(n)"
)
print(f"OK — {lora_trainable:,} LoRA adapter parameters injected")


# Classifier: LoRA backbone + linear head
class DinoV3LoRAClassifier(nn.Module):
    """DINOv3 backbone (LoRA-adapted) with a linear classification head."""

    def __init__(self, backbone, num_classes: int, embed_dim: int):
        super().__init__()
        self.backbone = backbone  # PEFT-wrapped timm encoder
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, images):
        features = self.backbone(images)  # (B, EMBED_DIM) — CLS token via timm
        return self.head(features)  # (B, num_classes)


lora_model = DinoV3LoRAClassifier(lora_backbone, NUM_CLASSES, EMBED_DIM).to(device)

total_trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print(f"Total trainable (LoRA adapters + head): {total_trainable:,}")

In [ ]:
# Helper functions — reusable train/evaluate loop (same pattern as Exercise 4)


def train_one_epoch(model, data_loader, criterion, optimizer, device):
    """Train for one epoch. Returns (avg_loss, accuracy)."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(data_loader, desc="Train", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, data_loader, criterion, device):
    """Evaluate on a dataset. Returns (avg_loss, accuracy)."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(data_loader, desc="Val", leave=False):
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)
    return running_loss / total, correct / total

### Task 3 — LoRA Training Loop

Train `lora_model` end-to-end for 5 epochs.  
Only the LoRA adapters and the classification head will be updated — the backbone weights stay frozen.

**Your task:**
- Use `nn.CrossEntropyLoss` and `Adam` with `lr=5e-4`  
  (lower than the linear probe — gentle updates to preserve pretrained knowledge)
- The optimizer should only update trainable parameters:  
  `filter(lambda p: p.requires_grad, lora_model.parameters())`
- Use `train_one_epoch` and `evaluate` from above
- Store metrics in `lora_history` dict with keys `train_loss`, `train_acc`, `val_loss`, `val_acc`
- Store final val accuracy in `lora_acc` and total time in `lora_time`

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
assert 0.0 < lora_acc <= 1.0, f"Accuracy should be between 0 and 1, got {lora_acc}"
assert len(lora_history["train_loss"]) == NUM_EPOCHS_LORA
print(f"OK — LoRA accuracy: {lora_acc:.1%}")

In [ ]:
epochs_lora = range(1, NUM_EPOCHS_LORA + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
_ = axes[0].plot(epochs_lora, lora_history["train_loss"], "o-", label="Train")
_ = axes[0].plot(epochs_lora, lora_history["val_loss"], "o-", label="Val")
_ = axes[0].set(xlabel="Epoch", ylabel="Loss", title="LoRA — Loss")
_ = axes[0].legend()
_ = axes[0].grid(alpha=0.3)
_ = axes[1].plot(epochs_lora, lora_history["train_acc"], "o-", label="Train")
_ = axes[1].plot(epochs_lora, lora_history["val_acc"], "o-", label="Val")
_ = axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="LoRA — Accuracy")
_ = axes[1].legend()
_ = axes[1].grid(alpha=0.3)
_ = plt.tight_layout()
plt.show()

**Question:** Why can LoRA outperform a linear probe on domain-shifted data even with so few trainable parameters?

<details>
<summary>Click to reveal answer</summary>

A linear probe operates on *fixed* features — it can rotate and scale the embedding space,  
but it cannot change *what the backbone attends to*.  
If the backbone is looking at the wrong things for histology (e.g., background textures  
instead of nuclear morphology), no linear layer can fix that.

LoRA lets the **attention layers adapt** — the QKV projections change which patches the  
transformer focuses on. With r=8 we add only a tiny rank-8 update to each QKV matrix,  
but that's enough to shift attention towards the features that matter for mitotic classification.

The key insight: **expressivity is in the right place**, not just in the number of parameters.  
A single well-placed low-rank update in attention can do what millions of downstream  
linear parameters cannot.

</details>

## Part 4 — Comparison

Let's compare both strategies side by side.

### Task 4 — Summary Table

Fill in `results` and print a formatted comparison table showing:  
strategy name, trainable parameters, val accuracy, and training time.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
assert len(results) == 2
print("OK")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names = [r[0] for r in results]
accs = [r[2] for r in results]
times = [r[3] for r in results]
colors = ["#4C72B0", "#C44E52"]

bars = axes[0].bar(names, accs, color=colors)
for bar, acc in zip(bars, accs, strict=False):
    _ = axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f"{acc:.1%}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )
_ = axes[0].set(ylabel="Val Accuracy", title="Accuracy Comparison", ylim=(0, 1.05))
_ = axes[0].grid(alpha=0.3, axis="y")

bars = axes[1].bar(names, times, color=colors)
for bar, t in zip(bars, times, strict=False):
    _ = axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{t:.0f}s",
        ha="center",
        va="bottom",
        fontweight="bold",
    )
_ = axes[1].set(ylabel="Time (s)", title="Training Time Comparison")
_ = axes[1].grid(alpha=0.3, axis="y")

_ = plt.suptitle("DINOv3 Adaptation on AMi-Br Microscopy", fontsize=13)
_ = plt.tight_layout()
plt.show()

## Discussion

**Q1: You have 50 labelled microscopy images and need a result today. Which strategy?**

<details>
<summary>Answer</summary>

**Linear probe.** With 50 images, LoRA still risks overfitting the adapter weights.  
The frozen features require zero training images to exist — only the linear head needs data.  
Train the linear probe, get a result in seconds.

</details>

---

**Q2: How does LoRA rank `r` affect the tradeoff?**

<details>
<summary>Answer</summary>

- Higher `r` → more adapter parameters → more capacity to adapt features → higher risk of overfitting on small datasets.
- Lower `r` → fewer params → less overfitting → but might underfit on large datasets with large domain gap.
- Typical classroom range: r=4 (minimal) to r=64 (aggressive).  
  For ~400 training images, r=4 or r=8 is safe.

</details>

---

**Q3: The next scale of DINOv3 is `vit_7b_patch16_dinov3.lvd1689m` (7 billion parameters).  
With LoRA r=8, roughly how many parameters would be trainable?**

<details>
<summary>Answer</summary>

With 7B parameters, a LoRA r=8 adapter on QKV layers adds roughly 5–10M trainable params —  
still less than 0.15% of the model. The same code works; you just change `MODEL_NAME`.  
Full fine-tuning of 7B params would require 56+ GB of GPU memory for weights alone;  
LoRA makes it feasible on a single A100.

</details>

---

## Summary

| Strategy | Trainable params | When to use |
|---|---|---|
| **Linear probe** | ~2K | Tiny datasets, same-domain tasks, instant results |
| **LoRA (r=8)** | ~1–2M | Domain shift, medium datasets, GPU available |
| **Partial fine-tune** | ~50M | Sufficient labelled data, moderate shift |
| **Full fine-tune** | all | Large labelled datasets, large domain gap |

**Key takeaway:** LoRA gives you the *right place* to adapt — inside the attention layers  
that build the model's representations — for a tiny fraction of the cost of full fine-tuning.